# IndoRad-VQA Dataset Audit & Feasibility Study

This notebook conducts a comprehensive dataset audit, exploratory data analysis (EDA), and feasibility study for the IndoRad-VQA dataset. The goal is to determine if this dataset is suitable for the research topic: **'Visual-Evidence-Based Reliability Under Language Shift: A Study of Medical VLMs on Indonesian Radiology VQA.'**

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from src import config, data_loader, schema_audit, data_quality, language_analysis
from src import bilingual_pairing, image_audit, text_analysis, leakage_audit
from src import statistical_analysis, visualization, feasibility

# Setup plotting style
visualization.setup_style()

## 1. Dataset Loading

In [2]:
# Load all splits from HuggingFace
print("Loading dataset...")
dfs = data_loader.load_all_splits()
split_summary = data_loader.get_split_summary(dfs)
display(split_summary)

# Combine for overall analysis
df_combined = pd.concat([df.assign(split=split_name) for split_name, df in dfs.items()], ignore_index=True)
print(f"Total rows combined: {len(df_combined)}")

Loading dataset...
Loading 'train' split...
Successfully loaded 'train' split directly via hf:// URL.
Loading 'test' split...
Successfully loaded 'test' split directly via hf:// URL.


,Split,Rows,Columns
0,train,1793,17
1,test,451,17


Total rows combined: 2244


## 2. Schema Audit

In [3]:
schema_results = {}
for split_name, df in dfs.items():
    schema_results[split_name] = schema_audit.audit_schema(df, split_name)
    print(f"\n--- {split_name.upper()} Schema ---")
    display(schema_results[split_name].head(config.MAX_DISPLAY_ROWS))

# Build data dictionary
data_dict = schema_audit.build_data_dictionary(dfs['train'])
schema_audit.save_schema_audit({'data_dictionary': data_dict}, config.PROCESSED_DIR)
display(data_dict)


--- TRAIN Schema ---


,column,dtype,non_null_count,null_count,null_pct,unique_count
0,qid,int32,1793,0,0.0,1793
1,image,object,1793,0,0.0,1793
2,image_name,object,1793,0,0.0,1793
3,question,object,1793,0,0.0,1583
4,question_indonesian,object,1793,0,0.0,1507
5,answer,object,1793,0,0.0,432
6,answer_indonesian,object,1793,0,0.0,410
7,answer_type,object,1793,0,0.0,2
8,question_type,object,0,1793,100.0,0
9,image_organ,object,0,1793,100.0,0



--- TEST Schema ---


,column,dtype,non_null_count,null_count,null_pct,unique_count
0,qid,int32,451,0,0.0,451
1,image,object,451,0,0.0,451
2,image_name,object,451,0,0.0,451
3,question,object,451,0,0.0,430
4,question_indonesian,object,451,0,0.0,418
5,answer,object,451,0,0.0,121
6,answer_indonesian,object,451,0,0.0,117
7,answer_type,object,451,0,0.0,2
8,question_type,object,0,451,100.0,0
9,image_organ,object,0,451,100.0,0


,Field Name,Data Type,Description,Example Value
0,qid,int32,Unique question identifier,0
1,image,object,Image binary or path object,{'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...
2,image_name,object,Filename or identifier of the radiology image,img_00000.png
3,question,object,English radiology question,are regions of the brain infarcted?
4,question_indonesian,object,Indonesian translated question,Apakah wilayah otak mengalami infark?
5,answer,object,English ground truth answer,yes
6,answer_indonesian,object,Indonesian translated ground truth answer,ya
7,answer_type,object,Question answer type category (CLOSED / OPEN),CLOSED
8,question_type,object,"Medical domain topic of question (organ, abnor...",None
9,image_organ,object,Target anatomical organ system,None


## 3. Data Quality Audit

In [4]:
quality_reports = {}
for split_name, df in dfs.items():
    quality_reports[split_name] = data_quality.generate_quality_report(df, split_name)
    print(f"\n--- {split_name.upper()} Missing Values ---")
    display(quality_reports[split_name]['missing_values'])
    print(f"\n--- {split_name.upper()} Duplicates ---")
    print(quality_reports[split_name]['duplicates'])

data_quality.save_quality_report(quality_reports['train'], config.PROCESSED_DIR / 'train')
data_quality.save_quality_report(quality_reports['test'], config.PROCESSED_DIR / 'test')


--- TRAIN Missing Values ---


,Column,Missing_Count,Missing_Pct
0,qid,0,0.0
1,image,0,0.0
2,image_name,0,0.0
3,question,0,0.0
4,question_indonesian,0,0.0
5,answer,0,0.0
6,answer_indonesian,0,0.0
7,answer_type,0,0.0
8,question_type,1793,100.0
9,image_organ,1793,100.0



--- TRAIN Duplicates ---
{'duplicate_english_questions': 210, 'duplicate_indonesian_questions': 286, 'exact_duplicate_rows': 0}

--- TEST Missing Values ---


,Column,Missing_Count,Missing_Pct
0,qid,0,0.0
1,image,0,0.0
2,image_name,0,0.0
3,question,0,0.0
4,question_indonesian,0,0.0
5,answer,0,0.0
6,answer_indonesian,0,0.0
7,answer_type,0,0.0
8,question_type,451,100.0
9,image_organ,451,100.0



--- TEST Duplicates ---
{'duplicate_english_questions': 21, 'duplicate_indonesian_questions': 33, 'exact_duplicate_rows': 0}


## 4. Language Analysis

In [5]:
lang_comparison = language_analysis.compare_language_distributions(dfs)
display(lang_comparison)

# Check text lengths
df_lengths = language_analysis.analyze_text_lengths(df_combined)
visualization.plot_question_length_distribution(df_lengths, config.FIGURES_DIR / 'question_length_dist.png')
visualization.plot_answer_length_distribution(df_lengths, config.FIGURES_DIR / 'answer_length_dist.png')

# Detect anomalies
anomalies = language_analysis.detect_language_anomalies(df_combined)
if not anomalies.empty:
    display(anomalies[['question', 'question_indonesian', 'answer', 'answer_indonesian', 'anomaly_reason']].head())

,Split,Total Rows,EN Available,EN %,ID Available,ID %
0,train,1793,1793,100.0,1793,100.0
1,test,451,451,100.0,451,100.0


## 5. Bilingual Pairing Audit (CRITICAL)

In [6]:
pairing_structures = {}
validation_dfs = []

for split_name, df in dfs.items():
    print(f"\n--- {split_name.upper()} Pairing Structure ---")
    struct = bilingual_pairing.analyze_pairing_structure(df)
    print(struct)
    pairing_structures[split_name] = struct
    
    # Validate pairs
    val_df = bilingual_pairing.validate_bilingual_pairs(df, split_name)
    validation_dfs.append(val_df)

val_combined = pd.concat(validation_dfs, ignore_index=True)
pairing_summary = bilingual_pairing.summarize_pairing(val_combined)

visualization.plot_bilingual_pair_validity(
    {k: v for k, v in pairing_summary.items() if k in ['valid', 'partial', 'invalid']}, 
    config.FIGURES_DIR / 'bilingual_validity.png'
)

# Sample pairs for review
sampled_pairs = bilingual_pairing.sample_pairs_for_review(val_combined)
display(sampled_pairs[['question_en', 'question_id', 'answer_en', 'answer_id', 'answer_match_type', 'pair_validity']])

bilingual_pairing.save_pairing_audit(val_combined, pairing_summary, config.MANIFESTS_DIR)


--- TRAIN Pairing Structure ---
{'total_rows': 1793, 'has_both_questions': 1793, 'has_both_answers': 1793, 'valid_pairs_count': 1793}

--- TEST Pairing Structure ---
{'total_rows': 451, 'has_both_questions': 451, 'has_both_answers': 451, 'valid_pairs_count': 451}


,question_en,question_id,answer_en,answer_id,answer_match_type,pair_validity
1666,what radiographic sign is seen in the intestines,Tanda radiografi apa yang terlihat pada usus?,air fluid level,tingkat cairan dan udara,different,valid
2239,is there subcutaneous air present in the right...,Apakah terdapat udara di bawah kulit pada lehe...,yes,ya,semantic_likely_equivalent,valid
1550,what is the modality of this mri?,Modality MRI ini apa?,t2,t2,exact_match,valid
240,is there bone in the top of the image?,Apakah ada tulang di bagian atas gambar?,no,tidak ada,semantic_likely_equivalent,valid
686,is there an aortic aneurysm?,Apakah ada aneurisma aorta?,no,tidak ada,semantic_likely_equivalent,valid
1882,what part of the body is being imaged here?,Bagian tubuh mana yang sedang di-imaging di sini?,abdomen,perut,different,valid
480,is the mass loculated?,Apakah massa tersebut terbelah menjadi beberap...,yes,ya,semantic_likely_equivalent,valid
668,is there swelling of the grey matter?,Apakah ada pembesaran pada jaringan grey matter?,yes,ya,semantic_likely_equivalent,valid
1655,is the left renal pelvis dilated?,Apakah pelvis ginjal kiri membesar?,no,tidak ada,semantic_likely_equivalent,valid
2002,name one organ that is captured by this image.,Sebutkan satu organ yang terlihat pada gambar ...,intestines,usus,different,valid


## 6. Image Audit

In [7]:
image_audits = {}
for split_name, df in dfs.items():
    print(f"\n--- {split_name.upper()} Image Audit ---")
    # Limiting check to first 100 images per split to save time in notebook, 
    # but actual study should check all.
    img_audit = image_audit.audit_images(df, split_name, max_check=None)
    image_audits[split_name] = img_audit
    
    stats = image_audit.compute_image_stats(img_audit)
    print(stats)
    
    image_audit.save_image_audit(img_audit, stats, config.MANIFESTS_DIR / split_name)

combined_img_audit = pd.concat(image_audits.values(), ignore_index=True)
img_stats = image_audit.compute_image_stats(combined_img_audit)


--- TRAIN Image Audit ---
{'total_records': 1793, 'unique_images': 1793, 'valid_images': np.int64(1793), 'valid_pct': np.float64(100.0)}

--- TEST Image Audit ---
{'total_records': 451, 'unique_images': 451, 'valid_images': np.int64(451), 'valid_pct': np.float64(100.0)}


## 7. Text & Question Analysis

In [8]:
q_types = text_analysis.analyze_question_types(df_combined)
a_types = text_analysis.analyze_answer_types(df_combined)
yes_no = text_analysis.check_yes_no_questions(df_combined)
ans_dist = text_analysis.analyze_answer_distribution(df_combined)

print("Yes/No Question Stats:")
print(yes_no)

visualization.plot_answer_type_distribution(df_combined, config.FIGURES_DIR / 'answer_types.png')

text_analysis.save_text_analysis({'yes_no': yes_no, 'answer_distribution': ans_dist}, config.PROCESSED_DIR)

Yes/No Question Stats:
{'total_questions': 2244, 'yes_no_count': 1191, 'yes_no_pct': np.float64(53.07), 'open_ended_count': 1053, 'open_ended_pct': np.float64(46.93)}


## 8. Leakage Audit

In [9]:
img_leakage = leakage_audit.check_image_leakage(dfs)
q_leakage = leakage_audit.check_question_leakage(dfs)
a_leakage = leakage_audit.check_answer_leakage(dfs)
pair_integrity = leakage_audit.check_bilingual_pair_split_integrity(dfs)

leakage_summary = leakage_audit.compute_leakage_summary(img_leakage, q_leakage, a_leakage)
print("Leakage Summary:")
print(leakage_summary)

if len(img_leakage) > 0:
    print("Found image leakage across splits. Generating recommended image-level split...")
    rec_split = leakage_audit.recommend_image_level_split(df_combined)
    display(rec_split.head())
else:
    print("No image leakage detected. Existing splits are clean at the image level.")
    rec_split = pd.DataFrame()

leakage_audit.save_leakage_audit(leakage_summary, rec_split, config.MANIFESTS_DIR)

Leakage Summary:
{'image_leakage_count': 0, 'question_leakage': {'question': {'count': 62, 'examples': ['are the lungs normal?', 'what pathology is seen in this image?', 'are the lungs hyperinflated?', 'what kind of image is this?', 'was this patient given iv contrast?']}, 'question_indonesian': {'count': 78, 'examples': ['apakah jantungnya terlalu besar?', 'apakah pasien ini mengalami pneumotoksis?', 'apakah diafragma bagian atas rata?', 'apakah ventrikel membesar?', 'di mana lokasi lesi otak?']}}, 'answer_leakage': {'question_answer': {'count': 42, 'examples': [['is the diaphragm obscured?', 'no'], ['what lobe of the brain is the lesion located in?', 'right frontal lobe'], ['is the liver normal?', 'yes'], ['is the liver normal?', 'no'], ['what is the plane of this image?', 'axial']]}}}
No image leakage detected. Existing splits are clean at the image level.


## 9. Research Feasibility Assessment

In [10]:
# Gather stats
split_stats = {'train': len(dfs['train']), 'test': len(dfs['test'])}

# Assess RQs
rq1 = feasibility.assess_rq1_feasibility(pairing_summary, leakage_summary, split_stats)
rq2 = feasibility.assess_rq2_feasibility(img_stats, pairing_summary)
rq3 = feasibility.assess_rq3_feasibility(yes_no, pairing_summary)
rq4 = feasibility.assess_rq4_feasibility(pairing_summary, leakage_summary, split_stats)

rq_assessments = {'rq1': rq1, 'rq2': rq2, 'rq3': rq3, 'rq4': rq4}

report_md = feasibility.generate_feasibility_report(rq_assessments)
presentation_md = feasibility.generate_presentation_summary(rq_assessments, pairing_summary, img_stats)

# Save reports
feasibility.save_feasibility_report(report_md + "\n\n" + presentation_md, config.REPORTS_DIR)

print("FEASIBILITY SUMMARY:")
for rq, assess in rq_assessments.items():
    print(f"{rq.upper()}: {assess['status']} - {assess['evidence']}")

FEASIBILITY SUMMARY:
RQ1: FEASIBLE - 100.0% valid bilingual Q&A pairs available with aligned English-Indonesian targets.
RQ2: FEASIBLE - Image integrity verified (100.0% valid images) for counterfactual perturbation experiments.
RQ3: FEASIBLE - 1191 Closed (Yes/No) questions available for exact logprob & confidence extraction.
RQ4: FEASIBLE - 1793 training samples sufficient for LoRA / QLoRA fine-tuning on local/cloud GPU.


# Corrective Audit — Image Identity and Translation Quality

Setelah EDA awal, audit korektif dilakukan untuk memverifikasi ulang (1) identitas gambar yang sesungguhnya melalui *pixel hashing*, dan (2) kualitas terjemahan pasangan *English-Indonesian* menggunakan *rule-based screening*.

Hasil lengkap audit korektif dapat dilihat pada output dari script run_corrective_audit.py
Script tersebut menghasilkan:
 1. Manifest identitas gambar (data/manifests/image_identity_manifest.csv)
 2. Manifest subset terjemahan high-confidence (data/manifests/high_confidence_bilingual_manifest.csv)
 3. Laporan kelayakan yang telah direvisi (reports/revised_feasibility_report.md)
 4. Visualisasi tambahan (figures/corrective_audit/)